# R2 · Calidad y gobierno de datos

**Responsable:** Dan (Rol B)  
**Revisor:** Leo (Rol D)

## Qué queda resuelto

- [x] Ejecutar las reglas de las 6 dimensiones (`src.calidad.reglas` y `data/clean/reglas_calidad.csv`)
- [x] Cuantificar: acumulados que bajan, días sin reporte, picos por rezago, fallecidos > casos, series vs diarios
- [x] Índice de calidad por país y top 10 de peores (P4)
- [x] Documentar las decisiones en `docs/decisiones_calidad.md`

In [ ]:
import sys, pathlib
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ))
import duckdb
import pandas as pd
from src.calidad.reglas import ejecutar_reglas
from src.calidad.indice import problemas_por_pais, indice_calidad

DW_PATH = RAIZ / 'data' / 'dw_covid.duckdb'
PARQUET_PATH = RAIZ / 'data' / 'clean' / 'reportes_unificados.parquet'

## Ejecutar las reglas de las 6 dimensiones (`src.calidad.reglas`)

In [ ]:
reglas_dw = pd.read_csv(RAIZ / 'data' / 'clean' / 'reglas_calidad.csv')
display(reglas_dw)

df_unif = pd.read_parquet(PARQUET_PATH, columns=['fecha_reporte', 'combined_key', 'country_region', 'lat', 'long', 'confirmed', 'deaths'])
display(ejecutar_reglas(df_unif))

## Cuantificar: acumulados que bajan, días sin reporte, picos por rezago, fallecidos > casos, series vs diarios

In [ ]:
con = duckdb.connect(str(DW_PATH), read_only=True)
resumen_problemas = con.execute('''
    SELECT
        COUNT(*) AS total_filas,
        SUM(CASE WHEN retrocede_casos THEN 1 ELSE 0 END) AS filas_retrocede_casos,
        SUM(CASE WHEN deaths > confirmed THEN 1 ELSE 0 END) AS filas_muertes_gt_casos
    FROM fact_covid
''').df()
display(resumen_problemas)
con.close()

## Índice de calidad por país y top 10 de peores (P4)

In [ ]:
indice_dw = pd.read_csv(RAIZ / 'data' / 'clean' / 'indice_calidad_paises.csv')
display(indice_dw.head(10))

## Decisiones documentadas en `docs/decisiones_calidad.md`

Se aplicó el principio **corregir, marcar o excluir; nunca imputar** (ver `docs/decisiones_calidad.md`).